package installation

In [1]:
# Install RDKit.
%%capture
!pip install rdkit-pypi

In [2]:

import torch
import numpy as np
from collections import defaultdict
import os
import timeit
import pickle
import sys

from rdkit import Chem
from rdkit.Chem import rdDepictor, Descriptors
from rdkit.Chem import MACCSkeys

from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics
from sklearn.datasets import make_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, auc, roc_curve, matthews_corrcoef, f1_score

### Check if GPU is available

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

Helper function

In [4]:
# dictionary of atoms where a new element gets a new index
def create_atoms(mol):
    atoms = [atom_dict[a.GetSymbol()] for a in mol.GetAtoms()]
    return np.array(atoms)

# format from_atomIDx : [to_atomIDx, bondDict]
def create_ijbonddict(mol):
    i_jbond_dict = defaultdict(lambda: [])
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bond = bond_dict[str(b.GetBondType())]
        i_jbond_dict[i].append((j, bond))
        i_jbond_dict[j].append((i, bond))
    return i_jbond_dict


def create_fingerprints(atoms, i_jbond_dict, radius):
    """Extract the r-radius subgraphs (i.e., fingerprints)
    from a molecular graph using WeisfeilerLehman-like algorithm."""

    if (len(atoms) == 1) or (radius == 0):
        fingerprints = [fingerprint_dict[a] for a in atoms]

    else:
        vertices = atoms
        for _ in range(radius):
            fingerprints = []
            for i, j_bond in i_jbond_dict.items():
                neighbors = [(vertices[j], bond) for j, bond in j_bond]
                fingerprint = (vertices[i], tuple(sorted(neighbors)))
                fingerprints.append(fingerprint_dict[fingerprint])
            vertices = fingerprints

    return np.array(fingerprints)


def create_adjacency(mol):
    adjacency  = Chem.GetAdjacencyMatrix(mol)
    n          = adjacency.shape[0]

    adjacency  = adjacency + np.eye(n)
    degree     = sum(adjacency)
    d_half     = np.sqrt(np.diag(degree))
    d_half_inv = np.linalg.inv(d_half)
    adjacency  = np.matmul(d_half_inv,np.matmul(adjacency,d_half_inv))
    return np.array(adjacency)


def dump_dictionary(dictionary, file_name):
    with open(file_name, 'wb') as f:
        pickle.dump(dict(dictionary), f)


def load_tensor(file_name, dtype):
    return [dtype(d).to(device) for d in np.load(file_name + '.npy', allow_pickle=True)]


def load_numpy(file_name):
    return np.load(file_name + '.npy', allow_pickle=True)


def load_pickle(file_name):
    with open(file_name, 'rb') as f:
        return pickle.load(f)


def shuffle_dataset(dataset, seed):
    np.random.seed(seed)
    np.random.shuffle(dataset)
    return dataset


def split_dataset(dataset, ratio):
    n = int(ratio * len(dataset))
    dataset_1, dataset_2 = dataset[:n], dataset[n:]
    return dataset_1, dataset_2

### Length of data

In [5]:
radius = 3
with open('Drugs.txt', 'r') as f:
    data_list = f.read().strip().split('\n')

"""Exclude the data contains "." in the smiles, which correspond to non-bonds"""
data_list = list(filter(lambda x: '.' not in x.strip().split()[0], data_list))
N = len(data_list)

print('Total number of drugs for prediction of target metabolic pathway : %d' %(N))


Total number of drugs for prediction of target metabolic pathway : 2196


### Read data

In [6]:

atom_dict = defaultdict(lambda: len(atom_dict))
bond_dict = defaultdict(lambda: len(bond_dict))
fingerprint_dict = defaultdict(lambda: len(fingerprint_dict))

Molecules, Adjacencies, Properties, MACCS_list   =  [], [], [], []

max_MolMR, min_MolMR     = -1000, 1000
max_MolLogP, min_MolLogP = -1000, 1000
max_MolWt, min_MolWt     = -1000, 1000
max_NumRotatableBonds, min_NumRotatableBonds = -1000, 1000
max_NumAliphaticRings, min_NumAliphaticRings = -1000, 1000
max_NumAromaticRings, min_NumAromaticRings   = -1000, 1000
max_NumSaturatedRings, min_NumSaturatedRings = -1000, 1000

for no, data in enumerate(data_list):

    print('/'.join(map(str, [no+1, N])))

    smiles, property_indices = data.strip().split('\t')
    property_s = property_indices.strip().split(',')

    property = np.zeros((1,7))
    for prop in property_s:
        property[0,int(prop)] = 1


    Properties.append(property)

    mol = Chem.MolFromSmiles(smiles)
    atoms = create_atoms(mol)
    i_jbond_dict = create_ijbonddict(mol)

    fingerprints = create_fingerprints(atoms, i_jbond_dict, radius)
    Molecules.append(fingerprints)

    adjacency = create_adjacency(mol)
    Adjacencies.append(adjacency)


    MACCS         = MACCSkeys.GenMACCSKeys(Chem.MolFromSmiles(smiles))
    MACCS_ids     = np.zeros((20,))
    MACCS_ids[0]  = Descriptors.MolMR(mol)
    MACCS_ids[1]  = Descriptors.MolLogP(mol)
    MACCS_ids[2]  = Descriptors.MolWt(mol)
    MACCS_ids[3]  = Descriptors.NumRotatableBonds(mol)
    MACCS_ids[4]  = Descriptors.NumAliphaticRings(mol)
    MACCS_ids[5]  = MACCS[108]
    MACCS_ids[6]  = Descriptors.NumAromaticRings(mol)
    MACCS_ids[7]  = MACCS[98]
    MACCS_ids[8]  = Descriptors.NumSaturatedRings(mol)
    MACCS_ids[9]  = MACCS[137]
    MACCS_ids[10] = MACCS[136]
    MACCS_ids[11] = MACCS[145]
    MACCS_ids[12] = MACCS[116]
    MACCS_ids[13] = MACCS[141]
    MACCS_ids[14] = MACCS[89]
    MACCS_ids[15] = MACCS[50]
    MACCS_ids[16] = MACCS[160]
    MACCS_ids[17] = MACCS[121]
    MACCS_ids[18] = MACCS[149]
    MACCS_ids[19] = MACCS[161]

    if max_MolMR < MACCS_ids[0]:
        max_MolMR = MACCS_ids[0]
    if min_MolMR > MACCS_ids[0]:
        min_MolMR = MACCS_ids[0]

    if max_MolLogP < MACCS_ids[1]:
        max_MolLogP = MACCS_ids[1]
    if min_MolLogP > MACCS_ids[1]:
        min_MolLogP = MACCS_ids[1]

    if max_MolWt < MACCS_ids[2]:
        max_MolWt = MACCS_ids[2]
    if min_MolWt > MACCS_ids[2]:
        min_MolWt = MACCS_ids[2]

    if max_NumRotatableBonds < MACCS_ids[3]:
        max_NumRotatableBonds = MACCS_ids[3]
    if min_NumRotatableBonds > MACCS_ids[3]:
        min_NumRotatableBonds = MACCS_ids[3]

    if max_NumAliphaticRings < MACCS_ids[4]:
        max_NumAliphaticRings = MACCS_ids[4]
    if min_NumAliphaticRings > MACCS_ids[4]:
        min_NumAliphaticRings = MACCS_ids[4]

    if max_NumAromaticRings < MACCS_ids[6]:
        max_NumAromaticRings = MACCS_ids[6]
    if min_NumAromaticRings > MACCS_ids[6]:
        min_NumAromaticRings = MACCS_ids[6]

    if max_NumSaturatedRings < MACCS_ids[8]:
        max_NumSaturatedRings = MACCS_ids[8]
    if min_NumSaturatedRings > MACCS_ids[8]:
        min_NumSaturatedRings = MACCS_ids[8]

    MACCS_list.append(MACCS_ids)

dir_input = ('Drug pathway/input'+str(radius)+'/')
os.makedirs(dir_input, exist_ok=True)

for n in range(N):
    for b in range(20):
        if b==0:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_MolMR)/(max_MolMR-min_MolMR)
        elif b==1:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_MolLogP)/(max_MolMR-min_MolLogP)
        elif b==2:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_MolWt)/(max_MolMR-min_MolWt)
        elif b==3:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_NumRotatableBonds)/(max_MolMR-min_NumRotatableBonds)
        elif b==4:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_NumAliphaticRings)/(max_MolMR-min_NumAliphaticRings)
        elif b==6:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_NumAromaticRings)/(max_MolMR-min_NumAromaticRings)
        elif b==8:
            MACCS_list[n][b] = (MACCS_list[n][b]-min_NumSaturatedRings)/(max_NumSaturatedRings-min_NumSaturatedRings)

np.save(dir_input + 'molecules', Molecules)
np.save(dir_input + 'adjacencies', Adjacencies)
np.save(dir_input + 'properties', Properties)
np.save(dir_input + 'maccs', np.asarray(MACCS_list))

dump_dictionary(fingerprint_dict, dir_input + 'fingerprint_dict.pickle')

print('End!')


1/2196
2/2196
3/2196
4/2196
5/2196
6/2196
7/2196
8/2196
9/2196
10/2196
11/2196
12/2196
13/2196
14/2196
15/2196
16/2196
17/2196
18/2196
19/2196
20/2196
21/2196
22/2196
23/2196
24/2196
25/2196
26/2196
27/2196
28/2196
29/2196
30/2196
31/2196
32/2196
33/2196
34/2196
35/2196
36/2196
37/2196
38/2196
39/2196
40/2196
41/2196
42/2196
43/2196
44/2196
45/2196
46/2196
47/2196
48/2196
49/2196
50/2196
51/2196
52/2196
53/2196
54/2196
55/2196
56/2196
57/2196
58/2196
59/2196
60/2196
61/2196
62/2196
63/2196
64/2196
65/2196
66/2196
67/2196
68/2196
69/2196
70/2196
71/2196
72/2196
73/2196
74/2196
75/2196
76/2196
77/2196
78/2196
79/2196
80/2196
81/2196
82/2196
83/2196
84/2196
85/2196
86/2196
87/2196
88/2196
89/2196
90/2196
91/2196
92/2196
93/2196
94/2196
95/2196
96/2196
97/2196
98/2196
99/2196
100/2196
101/2196
102/2196
103/2196
104/2196
105/2196
106/2196
107/2196
108/2196
109/2196
110/2196
111/2196
112/2196
113/2196
114/2196
115/2196
116/2196
117/2196
118/2196
119/2196
120/2196
121/2196
122/2196
123/2196
1

/usr/local/lib/python3.10/dist-packages/numpy/lib/npyio.py:518: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


### Data preparation

In [7]:
dir_input = ('Drug pathway/input'+str(radius)+'/')

molecules  = load_tensor(dir_input + 'molecules', torch.FloatTensor)
properties = load_numpy(dir_input + 'properties')
maccs      = load_numpy(dir_input + 'maccs')


with open(dir_input + 'fingerprint_dict.pickle', 'rb') as f:
    fingerprint_dict = pickle.load(f)

fingerprint_dict = load_pickle(dir_input + 'fingerprint_dict.pickle')
unknown          = 100
n_fingerprint    = len(fingerprint_dict) + unknown

my_maccs = []
for i in range(len(molecules)):
    target_mol = (n_fingerprint-1)*torch.ones([259], dtype=torch.float, device=device)
    target_mol[:molecules[i].size()[0]] = molecules[i]
    my_maccs.append(np.concatenate((target_mol.cpu().data.numpy(),maccs[i]), axis=0))

dataset = list(zip(properties, my_maccs))
dataset = shuffle_dataset(dataset, 4123)
dataset_train, dataset_   = split_dataset(dataset, 0.8)
dataset_dev, dataset_test = split_dataset(dataset_, 0.2)


data_batch = list(zip(*dataset_train))
properties_train, maccs_train = data_batch[-2], data_batch[-1]

data_batch = list(zip(*dataset_dev))
properties_dev, maccs_dev = data_batch[-2], data_batch[-1]

data_batch = list(zip(*dataset_test))
properties_test, maccs_test = data_batch[-2], data_batch[-1]


### Spliting data for traning & testing

In [8]:

train_len, dev_len, test_len = len(dataset_train), len(dataset_dev), len(dataset_test)

feature_len = maccs_train[0].shape[0]

X_train, X_dev, X_test = np.zeros((train_len,feature_len)), np.zeros((dev_len,feature_len)), np.zeros((test_len,feature_len))
Y_train, Y_dev, Y_test = np.zeros((train_len,7)), np.zeros((dev_len,7)), np.zeros((test_len,7))

for i in range(train_len):
    X_train[i,:] = maccs_train[i]
    Y_train[i] = properties_train[i][0]

for i in range(dev_len):
    X_dev[i,:]   = maccs_dev[i]
    Y_dev[i]   = properties_dev[i][0]

for i in range(test_len):
    X_test[i,:]  = maccs_test[i]
    Y_test[i]  = properties_test[i][0]

In [9]:
Y_train[:10]

array([[0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 0., 0.]])

In [10]:
X_train[:10]

array([[9.8610e+03, 9.8620e+03, 9.8630e+03, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [4.3530e+03, 6.2150e+03, 6.2160e+03, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [6.5280e+03, 6.5290e+03, 5.7550e+03, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00],
       ...,
       [5.4240e+03, 5.4250e+03, 5.4260e+03, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [2.2450e+03, 1.2583e+04, 1.1069e+04, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [6.5560e+03, 6.5550e+03, 6.5630e+03, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00]])

In [11]:
Y_test[:10]

array([[0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0., 0.]])

In [12]:
X_test[:10]

array([[1.0613e+04, 1.0614e+04, 1.0615e+04, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [3.9000e+02, 3.9100e+02, 3.9200e+02, ..., 1.0000e+00, 1.0000e+00,
        1.0000e+00],
       [7.0100e+02, 7.0100e+02, 7.0000e+02, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       ...,
       [6.1500e+02, 6.1500e+02, 1.9790e+03, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [1.1172e+04, 1.1171e+04, 1.1173e+04, ..., 1.0000e+00, 1.0000e+00,
        1.0000e+00],
       [4.6100e+02, 4.6200e+02, 3.9100e+03, ..., 0.0000e+00, 1.0000e+00,
        1.0000e+00]])

In [13]:
X_dev[:10]

array([[2.2380e+03, 1.3944e+04, 1.3945e+04, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [4.8800e+03, 4.8810e+03, 4.8800e+03, ..., 0.0000e+00, 1.0000e+00,
        1.0000e+00],
       [6.5370e+03, 1.1891e+04, 1.1892e+04, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       ...,
       [1.3240e+03, 1.3250e+03, 1.3240e+03, ..., 1.0000e+00, 1.0000e+00,
        1.0000e+00],
       [2.9700e+02, 2.9600e+02, 6.3320e+03, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [5.7070e+03, 6.0530e+03, 5.7070e+03, ..., 0.0000e+00, 1.0000e+00,
        1.0000e+00]])

In [14]:
Y_dev[:10]

array([[0., 0., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [1., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.]])

 Randomized Trees Classifier

In [15]:
# clf = ExtraTreesClassifier
# clf = ExtraTreesClassifier(n_estimators=500, criterion = 'entropy', bootstrap = False, max_features = 0.3, max_depth = 50, random_state=0)
# clf.fit(X_train, Y_train)
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier
clf = RandomForestClassifier(n_estimators=500, criterion = 'entropy', bootstrap = False, max_features = 0.3, max_depth = 50, random_state=0)
clf.fit(X_train, Y_train)

RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=50,
                       max_features=0.3, n_estimators=500, random_state=0)

In [16]:
# from sklearn.neighbors import KNeighborsClassifier
# clf = KNeighborsClassifier(n_neighbors=7)
# clf.fit(X_train, Y_train)

### Test set prediction accuracy

In [17]:
 Y_pred = clf.predict(X_test)
start = timeit.default_timer()
acc_score, prec_score, rec_score, mat_score, pr_score = 0., 0., 0., 0., 0.
for i in range(Y_test.shape[0]):
    acc_score  += accuracy_score(Y_test[i],Y_pred[i])
    prec_score += precision_score(Y_test[i],Y_pred[i])
    rec_score  += recall_score(Y_test[i],Y_pred[i])
    mat_score  += matthews_corrcoef(Y_test[i],Y_pred[i])
    pr_score   += f1_score(Y_test[i],Y_pred[i])



acc_score  = acc_score/Y_test.shape[0]
prec_score = prec_score/Y_test.shape[0]
rec_score  = rec_score/Y_test.shape[0]
mat_score  = mat_score/Y_test.shape[0]
pr_score  = pr_score/Y_test.shape[0]

end  = timeit.default_timer()
time = end - start

print('Accuracy \t Precision \t Recall \t Mattews \t F1 \t Time (sec)')
print('%.4f%% \t %.4f%% \t %.4f%% \t %.4f%% \t %.4f%% \t %.4f ' %(acc_score, prec_score, rec_score, mat_score, pr_score, time))

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defin

Accuracy 	 Precision 	 Recall 	 Mattews 	 F1 	 Time (sec)
0.9817% 	 0.9560% 	 0.9380% 	 0.9376% 	 0.9426% 	 1.9523 


In [18]:
import matplotlib.pyplot as plt
loss = clf.clf['loss']
val_loss = clf.clf['val_loss']
epochs = range(1, len(loss)+1)
plt.plot(epochs, loss, 'b', label ='Training loss')
plt.plot(epochs, val_loss, 'r', label ='Validation loss')
plt.title('Training Validation loss')
plt.xlabel('epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

AttributeError: ignored

### prediction report of target metabolic pathways

In [ ]:

print(metrics.classification_report(Y_test, Y_pred))
